# 24 TPTS：令牌传递与任务交换


本课按百科条目写：先定义……调度台在最后，可跳过。

---

## 1. 定义

**Token Passing with Task Swaps**（TPTS，Ma 等，终身 MAPF / 在线取送，AAMAS 2017 一带）：任务以流的形式到达，不是一次 n 对 n。系统里有一张 **令牌** = 规划权。

规则：

1. **空闲** 机器人拿到令牌后，在剩余任务里领 **对自己当前位姿最近** 的一个（最近 = 有向 A\* / Dijkstra 代价），变为执行中。
2. 所有「手上有任务」的车一起做 **HCA\***（优先规划，11 课）：按顺序时间 A\*，前人路径写入占用表。
3. 若 HCA 失败且 `allow_swap` 且恰好两车对头： **交换两人的任务** 再 HCA 一次。走廊两端互换目标时，交换后每人已经站在新目标上，冲突消失。

- **输入**：图 G；`robots`；当前位姿 `starts`；任务流 `task_stream = [(name, goal), ...]`；开关 `allow_swap`。
- **输出**：是否全部完成、完成顺序、发生过几次对头/交换。

它面向 **终身** 场景：做完一个再领下一个，位姿要随着路径更新。一次性匈牙利管不了「做完之后人在哪」。

---

## 2. 和谁相邻

|  | TPTS | 匈牙利 / 拍卖 | HCA\* / WHCA | CBS-TA |
|--|------|----------------|---------------|--------|
| 任务模型 | 流、可终身 | 一批静态 | 目标已给定 | 一批静态 + 最优路径 |
| 走路 | 有，HCA | 无 | 有 | CBS |
| 对头 | 可选 swap | 无 | 失败或绕路等待 | 约束树 |
| 最优 | 否 | 匹配层 / 在线 | 否 | 联合 SOC |

令牌传递（不带 swap 的 TP）已经能跑大型仓库启发式；swap 专门打「两车在窄廊对穿」这种 HCA 死结。

---

## 3. 不变量 / 定理

- 只有 `idle` 里的车能领任务；执行中的车不重新招标（和拍卖的 free 类似，但领的是 **对自己最近**，不是「刚到达的那一个」）。
- 领完后 `remaining` 去掉该任务名，避免两人领同一单。
- HCA 以 **当前** `pos[r]` 为起点，不是出生点。走完一截必须 `pos[r] = path[-1]`。
- 到达 `pos[r] == assign[r][1]` 则任务完成，车回到 idle。
- swap 交换的是 `assign` 里的 (name, goal) 对，不是物理传送。下一轮 HCA 仍从当前 pos 出发。

**没有完备、没有最优。** 对头且 `allow_swap=False` 时本实现直接停。若到达条件没把 `assign` 置空，while 会转无限圈——这是第 9 节的 bug。

走廊定理（教学）：A—C1—C2—C3—B 是一条宽 1 的无向廊。r0 在 A 要去 B，r1 在 B 要去 A，HCA 无论谁优先都会在廊里堵死（后车被前车占用表挡住且无法侧让）。交换目标后，r0 的目标变成 A（已在）、r1 的目标变成 B（已在），HCA 返回原地路径，SOC=0。

---

## 4. 完整伪代码（与实现对应）

```
function SwapIfHeadon(G, pos, assign):
    active ← {r: assign[r].goal for r in assign if assign[r]}
    plan ← HCA(G, pos_of_active, active)
    print before
    if plan.found:
        return plan, assign
    if not allow_swap or |active| != 2:
        return FAILURE
    a, b ← 两车
    assign[a], assign[b] ← assign[b], assign[a]
    active ← 交换后的目标
    plan ← HCA(G, pos, active)
    print after
    return plan, assign

function TPTS(G, robots, starts, stream, allow_swap):
    idle ← copy(robots);  pos ← copy(starts)
    remaining ← copy(stream);  assign[r] ← None
    while remaining 非空 or 有人在执行:
        for r in idle:
            领 remaining 里对 pos[r] 最近的任务
            idle.remove(r)
        plan ← HCA；失败则按上面 swap 一次
        沿 plan.paths 更新 pos；到达则 assign=None、回 idle
```

本课 **自己写** 的是中间那块：已经有两个目标、HCA 失败、交换再试。完整终身循环看库 `tpts`。

---

## 5. 复杂度

每领一次任务：对每个剩余任务一次 A\*，\(O(|T_{\mathrm{left}}|\cdot T_{A^*})\)。每次规划一次 HCA（各车一次时间 A\*）。swap 最多再加一次 HCA。终身则与任务流长度成正比。无指数搜索，也无最优证明。

---

## 6. 完整手算 / 小表

走廊 `corridor_headon`：

```
A -- C1 -- C2 -- C3 -- B
```

边权 1、双向。r0 在 A，r1 在 B。

**强制对穿（本课手写 swap 的输入）：** 目标 r0→B、r1→A。

| 步骤 | 内容 |
|------|------|
| HCA 先 r0 | r0 路径 A,C1,C2,C3,B，占用整廊 |
| 再 r1 | r1 从 B 出发，前方全被占，时间 A\* 失败 |
| found | False，paths 往往只有先规划的那辆 |
| 交换 | r0 目标改 A，r1 目标改 B |
| 再 HCA | r0 已在 A → `['A']`；r1 已在 B → `['B']`；SOC=0 |

**库 TPTS 领最近：** 流 `[("t0","B"),("t1","A")]`。

| 车 | 当前位置 | 剩余任务代价 | 领取 |
|----|----------|--------------|------|
| r0 | A | t0@B=4，t1@A=0 | **t1**（已经站在目标） |
| r1 | B | 只剩 t0@B=0 | **t0** |

两人各领了自己脚下的任务，HCA 不需要对穿，`headon=0`，`allow_swap` 真假都 `found=True`。教学含义：最近任务启发式 **有时已经避免对头**；swap 留给「两人最近的都是对面」那种流。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 7. 分步实现（自己写，print traces）

给定两个目标与一次失败的 HCA，交换再试。打印交换前后的目标、found、路径。


In [ ]:
from lib.maps import corridor_headon
from lib.algos.prioritized import hca

g = corridor_headon()
robots = ["r0", "r1"]
pos = {"r0": "A", "r1": "B"}
assign = {"r0": ("t0", "B"), "r1": ("t1", "A")}
print("走廊节点", sorted(g.nodes))
print("交换前 assign", assign)


def goals_of(assign):
    return {r: assign[r][1] for r in assign}


def swap_retry(graph, pos, assign):
    goals = goals_of(assign)
    plan = hca(graph, pos, goals)
    print("HCA 前 found=", plan["found"], "paths=", plan.get("paths"), "soc=", plan.get("soc"))
    if plan["found"]:
        return plan, assign
    rs = list(assign)
    print("HCA 失败，交换", rs[0], "<->", rs[1])
    assign = dict(assign)
    assign[rs[0]], assign[rs[1]] = assign[rs[1]], assign[rs[0]]
    print("交换后 assign", assign)
    plan2 = hca(graph, pos, goals_of(assign))
    print("HCA 后 found=", plan2["found"], "paths=", plan2.get("paths"), "soc=", plan2.get("soc"))
    return plan2, assign


plan, assign2 = swap_retry(g, pos, assign)
print("最终目标", goals_of(assign2), "成功", plan["found"])


## 8. 逐行实现表

| 行 | 作用 |
|----|------|
| `assign[r] = (name, goal)` | 名字用于完成列表，goal 给 HCA |
| `hca(graph, pos, goals)` | 起点必须是 **当前 pos**，不是出生点 |
| `if plan["found"]: return` | 能走就不换，避免无故打乱任务 |
| `rs = list(assign)` | 恰好两车才对头交换（与库 `len(active)==2` 一致） |
| 元组对调 | 换任务不换人、不瞬移 |
| 第二次 HCA | 仍从原 pos 出发；本例每人已在新目标上 |

库的终身循环还要：领最近、沿路径写 pos、到达后 `assign[r]=None` 回 idle。


In [ ]:
from lib.algos.assignment import tpts

stream = [("t0", "B"), ("t1", "A")]
starts = {"r0": "A", "r1": "B"}
print("tpts allow_swap=True ", tpts(g, robots, starts, stream, True))
print("tpts allow_swap=False", tpts(g, robots, starts, stream, False))
print("注意：领最近后两人已在目标上，headon 常为 0；对穿例子看上一格 swap_retry")


## 9. 常见 bug

1. **交换后仍用旧起点。** HCA 的 starts 必须是更新后的 `pos`。本例碰巧 pos 没变；走了几步再 swap 就会错。
2. **交换后不改 `assign` 只改局部 `goals`。** 下一轮又用旧任务，无限对头。
3. **到达不清空 assign。** `while remaining or any(assign)` 永远真，死循环。库在 `pos==goal` 时 `assign[r]=None; idle.append(r)`。
4. **swap 不限制两车。** 三人乱换没有对头语义。
5. **令牌发给忙碌车。** 一人两单，HCA 目标字典冲突。

## 10. 对照库

`tpts` 在 `lib/algos/assignment.py`：idle 领最近 → HCA → 失败且 `allow_swap` 则交换一次 → 用路径末点更新 pos。eval `test_24` 要求至少一种开关能完成，且 swap 的 headon 不比不换更糟。


In [ ]:
from lib.studio.widget import PlannerStudio

studio = PlannerStudio()
studio.show_multi(g.to_studio(), plan, "TPTS-swap")
studio


## 条目小结

| 项目 | 内容 |
|------|------|
| 场景 | 终身 / 任务流 MAPF |
| 令牌 | 空闲车的规划权；领对自己最近的剩余任务 |
| 行走 | 当前有任务的车跑 HCA |
| swap | 两车对头失败则换目标再试 |
| 走廊 | 对穿 → 交换 → 各留在原地 |

## 练习

1. 把强制对穿的目标写成 r0→B、r1→A，不交换时 HCA 的 `found` 是什么？交换后路径应是什么？
2. 若 `allow_swap=False` 且 HCA 失败，库选择 `break`。改成「原地等待一拍再试」会怎样？可能活，也可能永远等。
3. 为什么领最近有时让本课这条流根本不触发 swap？换一条流使两人最近的都是对面。
